# 01 · Exploración del feed de USGS (`all_day.geojson`)

**Objetivo:** conocer la fuente de datos recientes antes de procesarla: estructura, campos, tipos,
unidades, valores ausentes y casos especiales. Es la evidencia del criterio **Extracción**
de la rúbrica («Carga + análisis de tipos»).

**Datos:** `docs/material-profesor/ejemplos/feed-all_day-2026-10-28.geojson`
(una sola respuesta del feed, generada el 28/10/2026 a las 11:36 UTC).

> ⚠️ Es **una instantánea**: las cifras describen esta respuesta concreta, no el feed en general.

In [12]:
import json
from pathlib import Path

import pandas as pd

RUTA = Path("..") / "docs" / "material-profesor" / "ejemplos" / "feed-all_day-2026-10-28.geojson"

with open(RUTA, encoding="utf-8") as f:
    feed = json.load(f)   # el archivo completo como diccionario de Python

print("Pandas:", pd.__version__)
print("Claves de primer nivel:", list(feed.keys()))
feed["metadata"]

Pandas: 3.0.0
Claves de primer nivel: ['type', 'metadata', 'features', 'bbox']


{'generated': 1793187366000,
 'url': 'https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_day.geojson',
 'title': 'USGS All Earthquakes, Past Day',
 'status': 200,
 'api': '2.8.2',
 'count': 187}

## 1. Estructura de un evento

El feed es un `FeatureCollection`. Cada elemento de `features` es un terremoto con
`id`, `geometry` (punto) y `properties` (atributos). Veamos el primero tal cual llega.

In [13]:
print("Número de eventos:", len(feed["features"]), "| metadata.count:", feed["metadata"]["count"])
feed["features"][0]

Número de eventos: 187 | metadata.count: 187


{'type': 'Feature',
 'properties': {'mag': 0.75,
  'place': '5 km NW of The Geysers, CA',
  'time': 1793186692000,
  'updated': 1793186787595,
  'tz': None,
  'url': 'https://earthquake.usgs.gov/earthquakes/eventpage/nc75458882',
  'detail': 'https://earthquake.usgs.gov/earthquakes/feed/v1.0/detail/nc75458882.geojson',
  'felt': None,
  'cdi': None,
  'mmi': None,
  'alert': None,
  'status': 'automatic',
  'tsunami': 0,
  'sig': 9,
  'net': 'nc',
  'code': '75458882',
  'ids': ',nc75458882,',
  'sources': ',nc,',
  'types': ',nearby-cities,origin,phase-data,',
  'nst': 8,
  'dmin': 0.007189,
  'rms': 0.03,
  'gap': 124,
  'magType': 'md',
  'type': 'earthquake',
  'title': 'M 0.8 - 5 km NW of The Geysers, CA'},
 'geometry': {'type': 'Point',
  'coordinates': [-122.795166015625, 38.8151664733887, 2.74000000953674]},
 'id': 'nc75458882'}

## 2. Aplanar a una tabla

`pd.json_normalize` convierte cada evento en una fila y cada campo anidado en una columna
(`properties.mag`, `geometry.coordinates`…). Después separamos las coordenadas, que llegan
como lista **[longitud, latitud, profundidad_km]** (en ese orden).

In [14]:
df = pd.json_normalize(feed["features"])

coords = pd.DataFrame(df["geometry.coordinates"].tolist(), columns=["lon", "lat", "depth_km"])
df = pd.concat([df.drop(columns=["geometry.coordinates"]), coords], axis=1)

# OJO: hay DOS campos "type". El del Feature (siempre "Feature") y properties.type
# ("earthquake", "explosion"...). Si solo quitamos el prefijo, chocan. Los renombramos antes.
df = df.rename(columns={"type": "feature_type", "geometry.type": "geom_type"})

# Quitamos el prefijo "properties." para leer mejor
df.columns = [c.replace("properties.", "") for c in df.columns]
print("¿Nombres de columna únicos?", df.columns.is_unique)
print(df.shape)
df.head()

¿Nombres de columna únicos? True
(187, 32)


,feature_type,id,mag,place,time,updated,tz,url,detail,felt,...,dmin,rms,gap,magType,type,title,geom_type,lon,lat,depth_km
0,Feature,nc75458882,0.75,"5 km NW of The Geysers, CA",1793186692000,1793186787595,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/earthquakes/feed/v...,NaN,...,0.007189,0.0300,124.0,md,earthquake,"M 0.8 - 5 km NW of The Geysers, CA",Point,-122.795166,38.815166,2.7400
1,Feature,aka2026vizekq,1.30,"56 km E of McCarthy, Alaska",1793186283753,1793186382568,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/earthquakes/feed/v...,NaN,...,0.300000,0.5000,117.0,ml,earthquake,"M 1.3 - 56 km E of McCarthy, Alaska",Point,-141.881000,61.343000,5.0000
2,Feature,nn00926354,2.05,"12 km NNW of Silver Peak, Nevada",1793186144407,1793186299418,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/earthquakes/feed/v...,NaN,...,0.443000,0.3047,99.0,ml,earthquake,"M 2.1 - 12 km NNW of Silver Peak, Nevada",Point,-117.705600,37.847900,2.1122
3,Feature,aka2026viylbm,3.00,"25 km ESE of Kasilof, Alaska",1793184977538,1793185531613,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/earthquakes/feed/v...,1.0,...,0.500000,0.9000,79.0,ml,earthquake,"M 3.0 - 25 km ESE of Kasilof, Alaska",Point,-150.857000,60.243000,70.5000
4,Feature,aka2026viyhxm,2.20,"16 km SE of Denali National Park, Alaska",1793184730771,1793184861645,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/earthquakes/feed/v...,NaN,...,0.300000,0.8000,74.0,ml,earthquake,"M 2.2 - 16 km SE of Denali National Park, Alaska",Point,-151.477000,63.440000,5.0000


## 3. Tipos de datos y completitud por campo

Para cada columna: tipo detectado por pandas, cuántos valores hay y qué porcentaje está informado.
Esto es la dimensión **Completitud** de la sesión 7.

In [15]:
resumen = pd.DataFrame({
    "tipo_pandas": df.dtypes.astype(str),
    "informados": df.notna().sum(),
    "nulos": df.isna().sum(),
    "%_informado": (df.notna().mean() * 100).round(1),
}).sort_values("%_informado")
resumen

,tipo_pandas,informados,nulos,%_informado
tz,object,0,187,0.0
alert,str,6,181,3.2
mmi,float64,9,178,4.8
cdi,float64,16,171,8.6
felt,float64,16,171,8.6
feature_type,str,187,0,100.0
mag,float64,187,0,100.0
id,str,187,0,100.0
url,str,187,0,100.0
updated,int64,187,0,100.0


## 4. Unicidad dentro de una respuesta

In [16]:
print("Filas:", len(df), "| id únicos:", df["id"].nunique())

Filas: 187 | id únicos: 187


## 5. Tipo de evento (`type`)

El enunciado exige quedarnos con `type == "earthquake"` y **conservar el motivo de exclusión** del resto.

In [17]:
df["type"].value_counts()

type
earthquake    185
explosion       2
Name: count, dtype: int64

In [18]:
df.loc[df["type"] != "earthquake", ["id", "type", "mag", "depth_km", "place"]]

,id,type,mag,depth_km,place
114,uw714129552,explosion,0.40,-0.54,"9 km ENE of Hoodsport, Washington"
129,uw714129332,explosion,2.15,-0.62,"15 km S of Princeton, Canada"


## 6. Magnitud: rango, tipos de escala y umbral ≥ 4

In [19]:
print(df["mag"].describe())
print("\nEventos con mag >= 4:", (df["mag"] >= 4).sum())
df["magType"].value_counts()

count    187.000000
mean       2.009223
std        1.401058
min        0.220000
25%        0.975000
50%        1.700000
75%        2.355000
max        6.000000
Name: mag, dtype: float64

Eventos con mag >= 4: 28


magType
ml     108
md      51
mb      22
mww      6
Name: count, dtype: int64

## 7. Profundidad y coordenadas

Comprobamos rangos válidos (lat ∈ [−90, 90], lon ∈ [−180, 180]) y miramos las profundidades negativas.
**No se rechazan automáticamente** (enunciado §6): hay que estudiarlas.

In [20]:
print(df["geom_type"].value_counts())
print()
fuera_rango = df[(df["lat"].abs() > 90) | (df["lon"].abs() > 180)]
print("Coordenadas fuera de rango:", len(fuera_rango))
print(df["depth_km"].describe())
df.loc[df["depth_km"] < 0, ["id", "type", "mag", "depth_km", "place"]]

geom_type
Point    187
Name: count, dtype: int64

Coordenadas fuera de rango: 0
count    187.000000
mean      29.716328
std       52.996813
min       -2.310000
25%        5.000000
50%       10.000000
75%       30.560000
max      509.065000
Name: depth_km, dtype: float64


,id,type,mag,depth_km,place
114,uw714129552,explosion,0.40,-0.54,"9 km ENE of Hoodsport, Washington"
129,uw714129332,explosion,2.15,-0.62,"15 km S of Princeton, Canada"
185,uw714128982,earthquake,0.22,-2.31,"26 km ENE of Ashford, Washington"


## 8. Tiempos: ocurrencia, actualización y antigüedad

`time` y `updated` llegan en **milisegundos desde 1970 (Unix epoch)**. Los convertimos a fechas UTC.
- `updated − time`: cuánto tarda USGS en publicar la versión que vemos.
- `generated − time`: antigüedad del evento respecto a la respuesta (debería ser ≤ 24 h).

In [21]:
df["time_utc"] = pd.to_datetime(df["time"], unit="ms", utc=True)
df["updated_utc"] = pd.to_datetime(df["updated"], unit="ms", utc=True)
generado = pd.to_datetime(feed["metadata"]["generated"], unit="ms", utc=True)

df["retraso_min"] = (df["updated_utc"] - df["time_utc"]).dt.total_seconds() / 60
df["antiguedad_h"] = (generado - df["time_utc"]).dt.total_seconds() / 3600

print("Respuesta generada:", generado)
df[["retraso_min", "antiguedad_h"]].describe().round(2)

Respuesta generada: 2026-10-28 11:36:06+00:00


,retraso_min,antiguedad_h
count,187.00,187.00
mean,87.78,12.84
std,182.37,7.00
min,0.55,0.19
25%,1.63,7.85
50%,3.57,12.83
75%,78.08,19.44
max,925.80,23.99


## 9. Estado de revisión e identificadores asociados

- `status`: `automatic` (puede revisarse después → nuevas versiones) o `reviewed`.
- `ids`: lista de identificadores de distintas redes para el mismo evento. Si hay más de uno,
  el evento se ha asociado entre redes y el `id` preferido podría cambiar.

In [22]:
print(df["status"].value_counts(), "\n")
df["n_ids"] = df["ids"].str.strip(",").str.split(",").str.len()
print("Eventos con más de un id asociado:", (df["n_ids"] > 1).sum())
df.loc[df["n_ids"] > 1, ["id", "ids", "status", "mag"]].head(10)

status
automatic    120
reviewed      67
Name: count, dtype: int64 

Eventos con más de un id asociado: 8


,id,ids,status,mag
13,pr2026301000,",us7000ttn0,pr2026301000,",reviewed,4.07
74,aka2026vieele,",aka2026vieele,us7000ttl0,",reviewed,3.50
88,hv75064596,",us7000ttkb,hv75064596,",reviewed,2.63
109,us7000ttju,",aka2026vhwvab,us7000ttju,",reviewed,4.20
121,hv75064501,",us7000ttip,hv75064501,",reviewed,2.54
131,aka2026vhpcrg,",aka2026vhpcrg,us7000tti8,",reviewed,3.60
144,aka2026vhmjgz,",aka2026vhmjgz,us7000ttgx,",automatic,2.80
181,tx2026vdtnxx,",us7000ttft,tx2026vdtnxx,",reviewed,2.60


## 10. Conclusiones de la exploración

*(Se rellenan tras ejecutar el notebook, con las cifras obtenidas.)*

- 